In [15]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

matches = pd.read_csv('../data/processed/features_v5.csv')

# Make sure data is chronological
matches['MatchDateTime'] = pd.to_datetime(matches['MatchDateTime'])
matches = matches.sort_values('MatchDateTime').reset_index(drop=True)

# --------------------------------------------------
# FEATURES
# --------------------------------------------------

# Everything after MatchDateTime
start_col = matches.columns.get_loc('MatchDateTime') + 1

features = list(matches.columns[start_col:])

# Remove anything that should NOT be used as a feature
exclude = [
    'FTR',
    'Season'
]

features = [
    feature for feature in features
    if feature not in exclude
]

print(f"Number of features: {len(features)}")
print(features)

Number of features: 53
['HomeGoalsLast5', 'AwayGoalsLast5', 'HomePointsLast5', 'AwayPointsLast5', 'HomeGoalsAgainstLast5', 'AwayGoalsAgainstLast5', 'HomeShotsForLast5', 'HomeShotsOnTargetLast5', 'HomeShotsAgainstLast5', 'HomeShotsOnTargetAgainstLast5', 'AwayShotsForLast5', 'AwayShotsOnTargetLast5', 'AwayShotsAgainstLast5', 'AwayShotsOnTargetAgainstLast5', 'HomeGamesPlayed', 'AwayGamesPlayed', 'HomePointsSeason', 'AwayPointsSeason', 'HomePPG', 'AwayPPG', 'HomeGoalsPerGame', 'AwayGoalsPerGame', 'HomeGoalsAgainstPerGame', 'AwayGoalsAgainstPerGame', 'HomeGDPerGame', 'AwayGDPerGame', 'PPGDiff', 'GDPerGameDiff', 'GoalsPerGameDiff', 'GoalsAgainstPerGameDiff', 'HomeHomePointsLast5', 'HomeHomeGoalsLast5', 'HomeHomeGoalsAgainstLast5', 'AwayAwayPointsLast5', 'AwayAwayGoalsLast5', 'AwayAwayGoalsAgainstLast5', 'HomeAwayPointsDiffLast5', 'HomeAwayGoalsDiffLast5', 'HomeAwayGoalsAgainstDiffLast5', 'GoalDiffLast5', 'PointsDiffLast5', 'GoalAgainstDiffLast5', 'ShotDiffLast5', 'ShotOTDiffLast5', 'HomeElo'

In [16]:
test_seasons = ['22-23', '23-24', '24-25', '25-26']

results = []

for test_season in test_seasons:

    # Everything before the test season
    train = matches[matches['Season'] < test_season]
    test = matches[matches['Season'] == test_season]

    y_train = train['FTR']
    y_test = test['FTR']

    for feature in features:

        # Drop rows where this feature is missing
        train_valid = train[[feature, 'FTR']].dropna()
        test_valid = test[[feature, 'FTR']].dropna()

        X_train = train_valid[[feature]]
        y_train_feature = train_valid['FTR']

        X_test = test_valid[[feature]]
        y_test_feature = test_valid['FTR']

        # Need at least two classes
        if y_train_feature.nunique() < 2:
            continue

        model = LogisticRegression(
            max_iter=1000
        )

        model.fit(X_train, y_train_feature)

        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)

        accuracy = accuracy_score(
            y_test_feature,
            preds
        )

        loss = log_loss(
            y_test_feature,
            probs,
            labels=model.classes_
        )

        results.append({
            'TestSeason': test_season,
            'Feature': feature,
            'Accuracy': accuracy,
            'LogLoss': loss
        })

results_df = pd.DataFrame(results)

results_df.head()

,TestSeason,Feature,Accuracy,LogLoss
0,22-23,HomeGoalsLast5,0.484211,1.036270
1,22-23,AwayGoalsLast5,0.478947,1.045792
2,22-23,HomePointsLast5,0.478947,1.027249
3,22-23,AwayPointsLast5,0.478947,1.046186
4,22-23,HomeGoalsAgainstLast5,0.478947,1.039459


In [17]:
accuracy_table = results_df.pivot(
    index='Feature',
    columns='TestSeason',
    values='Accuracy'
)

accuracy_table['MeanAccuracy'] = accuracy_table.mean(axis=1)

accuracy_table = accuracy_table.sort_values(
    'MeanAccuracy',
    ascending=False
)

print(accuracy_table)

TestSeason                        22-23     23-24     24-25     25-26  \
Feature                                                                 
EloDiff                        0.552632  0.557895  0.536842  0.484211   
EloDiffFeature                 0.552632  0.557895  0.536842  0.484211   
PPGDiff                        0.534211  0.555263  0.534211  0.478947   
GDPerGameDiff                  0.518421  0.576316  0.505263  0.489474   
GoalsPerGameDiff               0.523684  0.550000  0.481579  0.486842   
GoalsAgainstPerGameDiff        0.513158  0.550000  0.500000  0.471053   
HomeAwayShotsDiff              0.513158  0.544737  0.500000  0.455263   
ShotDiffLast5                  0.513158  0.544737  0.500000  0.455263   
HomeElo                        0.497368  0.518421  0.497368  0.471053   
HomeAwayPointsDiffLast5        0.513158  0.526316  0.484211  0.447368   
HomeAwayPPGDiff                0.513158  0.526316  0.484211  0.447368   
PointsDiffLast5                0.502632  0.534211  

In [18]:
logloss_table = results_df.pivot(
    index='Feature',
    columns='TestSeason',
    values='LogLoss'
)

logloss_table['MeanLogLoss'] = logloss_table.mean(axis=1)

logloss_table = logloss_table.sort_values(
    'MeanLogLoss',
    ascending=True
)

print(logloss_table)

TestSeason                        22-23     23-24     24-25     25-26  \
Feature                                                                 
EloDiffFeature                 0.999150  0.940447  0.997357  1.032826   
EloDiff                        0.999150  0.940447  0.997357  1.032826   
PPGDiff                        0.992474  0.968941  1.009410  1.057092   
GDPerGameDiff                  0.995117  0.968413  1.012659  1.054270   
HomeElo                        0.998290  0.984892  1.026030  1.063278   
GoalsPerGameDiff               1.002636  0.980070  1.045912  1.060374   
HomeAwayShotsDiff              1.023570  0.981842  1.031446  1.066713   
ShotDiffLast5                  1.023570  0.981842  1.031446  1.066713   
ShotOTDiffLast5                1.012958  0.984000  1.042702  1.074884   
HomeAwayShotsOTDiff            1.012958  0.984000  1.042702  1.074884   
PointsDiffLast5                1.019465  0.991109  1.045085  1.064819   
GoalsAgainstPerGameDiff        1.033330  1.003380  

In [19]:
results_df.to_csv(
    '../data/processed/feature_ablation_results.csv',
    index=False
)

accuracy_table.to_csv(
    '../data/processed/feature_ablation_accuracy.csv'
)

logloss_table.to_csv(
    '../data/processed/feature_ablation_logloss.csv'
)

print("Saved feature ablation results")

Saved feature ablation results


In [20]:
all_features = features.copy()

In [21]:
ablation_results = []

for test_season in test_seasons:

    train = matches[matches['Season'] < test_season]
    test = matches[matches['Season'] == test_season]

    for removed_feature in all_features:

        model_features = [
            f for f in all_features
            if f != removed_feature
        ]

        # Only use rows without missing values
        train_valid = train[
            model_features + ['FTR']
        ].dropna()

        test_valid = test[
            model_features + ['FTR']
        ].dropna()

        X_train = train_valid[model_features]
        y_train = train_valid['FTR']

        X_test = test_valid[model_features]
        y_test = test_valid['FTR']

        model = LogisticRegression(
            max_iter=2000
        )

        model.fit(X_train, y_train)

        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)

        accuracy = accuracy_score(
            y_test,
            preds
        )

        loss = log_loss(
            y_test,
            probs,
            labels=model.classes_
        )

        ablation_results.append({
            'TestSeason': test_season,
            'RemovedFeature': removed_feature,
            'Accuracy': accuracy,
            'LogLoss': loss
        })

ablation_df = pd.DataFrame(ablation_results)

c:\Users\harry\football-analysis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\harry\football-analysis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/p

In [22]:
full_results = []

for test_season in test_seasons:

    train = matches[matches['Season'] < test_season]
    test = matches[matches['Season'] == test_season]

    train_valid = train[all_features + ['FTR']].dropna()
    test_valid = test[all_features + ['FTR']].dropna()

    X_train = train_valid[all_features]
    y_train = train_valid['FTR']

    X_test = test_valid[all_features]
    y_test = test_valid['FTR']

    model = LogisticRegression(
        max_iter=2000
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)

    full_results.append({
        'TestSeason': test_season,
        'Accuracy': accuracy_score(y_test, preds),
        'LogLoss': log_loss(
            y_test,
            probs,
            labels=model.classes_
        )
    })

full_results_df = pd.DataFrame(full_results)

print(full_results_df)

c:\Users\harry\football-analysis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\harry\football-analysis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/p

  TestSeason  Accuracy   LogLoss
0      22-23  0.536842  1.069260
1      23-24  0.550000  0.969704
2      24-25  0.507895  1.024281
3      25-26  0.484211  1.054600


c:\Users\harry\football-analysis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [23]:
ablation_summary = []

for feature in all_features:

    feature_results = ablation_df[
        ablation_df['RemovedFeature'] == feature
    ]

    for season in test_seasons:

        full_score = full_results_df[
            full_results_df['TestSeason'] == season
        ].iloc[0]

        removed_score = feature_results[
            feature_results['TestSeason'] == season
        ].iloc[0]

        ablation_summary.append({
            'Feature': feature,
            'TestSeason': season,
            'FullAccuracy': full_score['Accuracy'],
            'WithoutFeatureAccuracy': removed_score['Accuracy'],
            'AccuracyChange': (
                removed_score['Accuracy']
                - full_score['Accuracy']
            ),
            'FullLogLoss': full_score['LogLoss'],
            'WithoutFeatureLogLoss': removed_score['LogLoss'],
            'LogLossChange': (
                removed_score['LogLoss']
                - full_score['LogLoss']
            )
        })

ablation_summary_df = pd.DataFrame(ablation_summary)

In [24]:
summary = (
    ablation_summary_df
    .groupby('Feature')
    .agg(
        MeanAccuracyChange=('AccuracyChange', 'mean'),
        MeanLogLossChange=('LogLossChange', 'mean')
    )
    .sort_values(
        'MeanLogLossChange',
        ascending=False
    )
)

print(summary)

                               MeanAccuracyChange  MeanLogLossChange
Feature                                                             
AwayAwayGoalsAgainstLast5           -3.947368e-03           0.013843
HomeShotsForLast5                   -7.894737e-03           0.009548
GoalsAgainstPerGameDiff             -1.315789e-03           0.008819
HomePPG                             -5.263158e-03           0.008333
AwayElo                             -7.236842e-03           0.007892
AwayAwayGoalsLast5                  -3.947368e-03           0.007485
HomeAwayShotsDiff                   -1.315789e-03           0.007110
AwayPointsLast5                     -5.921053e-03           0.007082
GoalAgainstDiffLast5                -3.289474e-03           0.006926
AwayShotsOnTargetLast5              -4.605263e-03           0.006695
HomeAwayGoalsAgainstDiffLast5       -6.578947e-04           0.006442
HomeElo                             -1.118421e-02           0.006344
HomeAwayShotsOTDiff               

In [25]:
candidate_features = [
    # Team strength
    "EloDiff",

    # Season performance
    "PPGDiff",
    "GDPerGameDiff",
    "GoalsPerGameDiff",
    "GoalsAgainstPerGameDiff",

    # Overall last-5 form
    "PointsDiffLast5",
    "GoalDiffLast5",
    "GoalAgainstDiffLast5",
    "ShotDiffLast5",
    "ShotOTDiffLast5",

    # Venue-specific last-5 form
    "HomeAwayPPGDiff",
    "HomeAwayGoalsDiff",
    "HomeAwayGoalsAgainstDiff",
    "HomeAwayShotsDiff",
    "HomeAwayShotsOTDiff",
]

In [26]:
corr = matches[candidate_features].corr()

print(corr.round(2))

                          EloDiff  PPGDiff  GDPerGameDiff  GoalsPerGameDiff  \
EloDiff                      1.00     0.79           0.76              0.67   
PPGDiff                      0.79     1.00           0.92              0.80   
GDPerGameDiff                0.76     0.92           1.00              0.87   
GoalsPerGameDiff             0.67     0.80           0.87              1.00   
GoalsAgainstPerGameDiff      0.61     0.75           0.82              0.43   
PointsDiffLast5              0.66     0.71           0.65              0.56   
GoalDiffLast5                0.55     0.61           0.65              0.72   
GoalAgainstDiffLast5         0.49     0.53           0.58              0.32   
ShotDiffLast5                0.58     0.52           0.55              0.50   
ShotOTDiffLast5              0.55     0.56           0.59              0.58   
HomeAwayPPGDiff              0.63     0.65           0.59              0.52   
HomeAwayGoalsDiff            0.54     0.56          

In [27]:
from itertools import combinations
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss


candidate_features = [
    # Team strength
    "EloDiff",

    # Season performance
    "PPGDiff",
    "GDPerGameDiff",
    "GoalsPerGameDiff",
    "GoalsAgainstPerGameDiff",

    # Overall last-5 form
    "PointsDiffLast5",
    "GoalDiffLast5",
    "GoalAgainstDiffLast5",
    "ShotDiffLast5",
    "ShotOTDiffLast5",

    # Venue-specific last-5 form
    "HomeAwayPPGDiff",
    "HomeAwayGoalsDiff",
    "HomeAwayGoalsAgainstDiff",
    "HomeAwayShotsDiff",
    "HomeAwayShotsOTDiff"
]


test_seasons = [
    "22-23",
    "23-24",
    "24-25",
    "25-26"
]


results = []


for n_features in range(1, 7):

    combinations_to_test = combinations(
        candidate_features,
        n_features
    )

    for feature_combination in combinations_to_test:

        season_accuracies = []
        season_log_losses = []

        for test_season in test_seasons:

            # Train on all previous seasons
            train = matches[
                matches["Season"] < test_season
            ]

            test = matches[
                matches["Season"] == test_season
            ]

            train_valid = train[
                list(feature_combination) + ["FTR"]
            ].dropna()

            test_valid = test[
                list(feature_combination) + ["FTR"]
            ].dropna()

            X_train = train_valid[
                list(feature_combination)
            ]

            y_train = train_valid["FTR"]

            X_test = test_valid[
                list(feature_combination)
            ]

            y_test = test_valid["FTR"]

            model = LogisticRegression(
                max_iter=2000
            )

            model.fit(
                X_train,
                y_train
            )

            preds = model.predict(X_test)

            probs = model.predict_proba(X_test)

            accuracy = accuracy_score(
                y_test,
                preds
            )

            loss = log_loss(
                y_test,
                probs,
                labels=model.classes_
            )

            season_accuracies.append(accuracy)
            season_log_losses.append(loss)

        results.append({
            "NumFeatures": n_features,
            "Features": " + ".join(feature_combination),

            "22-23 Accuracy": season_accuracies[0],
            "23-24 Accuracy": season_accuracies[1],
            "24-25 Accuracy": season_accuracies[2],
            "25-26 Accuracy": season_accuracies[3],

            "MeanAccuracy": np.mean(season_accuracies),

            "22-23 LogLoss": season_log_losses[0],
            "23-24 LogLoss": season_log_losses[1],
            "24-25 LogLoss": season_log_losses[2],
            "25-26 LogLoss": season_log_losses[3],

            "MeanLogLoss": np.mean(season_log_losses)
        })


results_df = pd.DataFrame(results)

print("Finished.")
print(f"Tested {len(results_df)} combinations.")

Finished.
Tested 9948 combinations.


In [28]:
best_accuracy = results_df.sort_values(
    "MeanAccuracy",
    ascending=False
)

print(
    best_accuracy[
        [
            "NumFeatures",
            "Features",
            "MeanAccuracy",
            "22-23 Accuracy",
            "23-24 Accuracy",
            "24-25 Accuracy",
            "25-26 Accuracy"
        ]
    ].head(20)
)

      NumFeatures                                           Features  \
2494            5  EloDiff + GoalsPerGameDiff + PointsDiffLast5 +...   
856             4  EloDiff + GoalDiffLast5 + GoalAgainstDiffLast5...   
861             4  EloDiff + GoalDiffLast5 + GoalAgainstDiffLast5...   
890             4  EloDiff + GoalAgainstDiffLast5 + ShotOTDiffLas...   
900             4  EloDiff + GoalAgainstDiffLast5 + HomeAwayGoals...   
2892            5  EloDiff + GoalAgainstDiffLast5 + ShotOTDiffLas...   
2825            5  EloDiff + GoalDiffLast5 + GoalAgainstDiffLast5...   
189             3  EloDiff + GoalAgainstDiffLast5 + HomeAwayShots...   
893             4  EloDiff + GoalAgainstDiffLast5 + ShotOTDiffLas...   
663             4  EloDiff + GDPerGameDiff + GoalsPerGameDiff + H...   
658             4  EloDiff + GDPerGameDiff + GoalsPerGameDiff + S...   
184             3   EloDiff + GoalAgainstDiffLast5 + ShotOTDiffLast5   
626             4  EloDiff + PPGDiff + GoalAgainstDiffLast5 + Sh

In [29]:
best_logloss = results_df.sort_values(
    "MeanLogLoss",
    ascending=True
)

print(
    best_logloss[
        [
            "NumFeatures",
            "Features",
            "MeanLogLoss",
            "22-23 LogLoss",
            "23-24 LogLoss",
            "24-25 LogLoss",
            "25-26 LogLoss"
        ]
    ].head(20)
)

      NumFeatures                                           Features  \
702             4  EloDiff + GDPerGameDiff + ShotDiffLast5 + Home...   
138             3            EloDiff + GDPerGameDiff + ShotDiffLast5   
143             3        EloDiff + GDPerGameDiff + HomeAwayShotsDiff   
2264            5  EloDiff + GDPerGameDiff + GoalsPerGameDiff + S...   
662             4  EloDiff + GDPerGameDiff + GoalsPerGameDiff + H...   
657             4  EloDiff + GDPerGameDiff + GoalsPerGameDiff + S...   
727             4  EloDiff + GoalsPerGameDiff + GoalsAgainstPerGa...   
722             4  EloDiff + GoalsPerGameDiff + GoalsAgainstPerGa...   
2474            5  EloDiff + GoalsPerGameDiff + GoalsAgainstPerGa...   
2234            5  EloDiff + GDPerGameDiff + GoalsPerGameDiff + G...   
2309            5  EloDiff + GDPerGameDiff + GoalsAgainstPerGameD...   
667             4  EloDiff + GDPerGameDiff + GoalsAgainstPerGameD...   
2229            5  EloDiff + GDPerGameDiff + GoalsPerGameDiff + 

In [30]:
results_df["AccuracyStd"] = results_df[
    [
        "22-23 Accuracy",
        "23-24 Accuracy",
        "24-25 Accuracy",
        "25-26 Accuracy"
    ]
].std(axis=1)

results_df["LogLossStd"] = results_df[
    [
        "22-23 LogLoss",
        "23-24 LogLoss",
        "24-25 LogLoss",
        "25-26 LogLoss"
    ]
].std(axis=1)

In [31]:
best_consistent = results_df.sort_values(
    ["MeanLogLoss", "LogLossStd"],
    ascending=[True, True]
)

print(
    best_consistent[
        [
            "NumFeatures",
            "Features",
            "MeanAccuracy",
            "AccuracyStd",
            "MeanLogLoss",
            "LogLossStd"
        ]
    ].head(20)
)

      NumFeatures                                           Features  \
702             4  EloDiff + GDPerGameDiff + ShotDiffLast5 + Home...   
138             3            EloDiff + GDPerGameDiff + ShotDiffLast5   
143             3        EloDiff + GDPerGameDiff + HomeAwayShotsDiff   
2264            5  EloDiff + GDPerGameDiff + GoalsPerGameDiff + S...   
662             4  EloDiff + GDPerGameDiff + GoalsPerGameDiff + H...   
657             4  EloDiff + GDPerGameDiff + GoalsPerGameDiff + S...   
727             4  EloDiff + GoalsPerGameDiff + GoalsAgainstPerGa...   
722             4  EloDiff + GoalsPerGameDiff + GoalsAgainstPerGa...   
2474            5  EloDiff + GoalsPerGameDiff + GoalsAgainstPerGa...   
2234            5  EloDiff + GDPerGameDiff + GoalsPerGameDiff + G...   
2309            5  EloDiff + GDPerGameDiff + GoalsAgainstPerGameD...   
667             4  EloDiff + GDPerGameDiff + GoalsAgainstPerGameD...   
2229            5  EloDiff + GDPerGameDiff + GoalsPerGameDiff + 

In [32]:
feature_groups = {
    "strength": [
        "EloDiff",
        "PPGDiff",
        "GDPerGameDiff"
    ],

    "attack_form": [
        "GoalDiffLast5",
        "ShotDiffLast5",
        "ShotOTDiffLast5"
    ],

    "defence_form": [
        "GoalAgainstDiffLast5"
    ],

    "season_attack": [
        "GoalsPerGameDiff"
    ],

    "season_defence": [
        "GoalsAgainstPerGameDiff"
    ],

    "venue_form": [
        "HomeAwayPPGDiff",
        "HomeAwayGoalsDiff",
        "HomeAwayGoalsAgainstDiff"
    ]
}

In [33]:
from itertools import product
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss


feature_groups = {
    "strength": [
        "EloDiff",
        "PPGDiff",
        "GDPerGameDiff"
    ],

    "attack_form": [
        "GoalDiffLast5",
        "ShotDiffLast5",
        "ShotOTDiffLast5"
    ],

    "defence_form": [
        "GoalAgainstDiffLast5"
    ],

    "season_attack": [
        "GoalsPerGameDiff"
    ],

    "season_defence": [
        "GoalsAgainstPerGameDiff"
    ],

    "venue_form": [
        "HomeAwayPPGDiff",
        "HomeAwayGoalsDiff",
        "HomeAwayGoalsAgainstDiff"
    ]
}


# Include "None" so a group doesn't have to contribute a feature
options = []

for group, features_group in feature_groups.items():
    options.append([None] + features_group)


combinations_to_test = []

for combination in product(*options):

    selected = [
        feature
        for feature in combination
        if feature is not None
    ]

    # Require at least 2 features
    if len(selected) >= 2:
        combinations_to_test.append(selected)


print(
    f"Testing {len(combinations_to_test)} combinations"
)

Testing 499 combinations


In [34]:
results = []

test_seasons = [
    "22-23",
    "23-24",
    "24-25",
    "25-26"
]


for selected_features in combinations_to_test:

    accuracies = []
    log_losses = []

    for test_season in test_seasons:

        train = matches[
            matches["Season"] < test_season
        ]

        test = matches[
            matches["Season"] == test_season
        ]

        train_valid = train[
            selected_features + ["FTR"]
        ].dropna()

        test_valid = test[
            selected_features + ["FTR"]
        ].dropna()

        X_train = train_valid[selected_features]
        y_train = train_valid["FTR"]

        X_test = test_valid[selected_features]
        y_test = test_valid["FTR"]

        model = LogisticRegression(
            max_iter=2000
        )

        model.fit(X_train, y_train)

        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)

        accuracies.append(
            accuracy_score(y_test, preds)
        )

        log_losses.append(
            log_loss(
                y_test,
                probs,
                labels=model.classes_
            )
        )

    results.append({
        "NumFeatures": len(selected_features),
        "Features": " + ".join(selected_features),

        "MeanAccuracy": np.mean(accuracies),
        "AccuracyStd": np.std(accuracies),

        "MeanLogLoss": np.mean(log_losses),
        "LogLossStd": np.std(log_losses),

        "22-23 Accuracy": accuracies[0],
        "23-24 Accuracy": accuracies[1],
        "24-25 Accuracy": accuracies[2],
        "25-26 Accuracy": accuracies[3],

        "22-23 LogLoss": log_losses[0],
        "23-24 LogLoss": log_losses[1],
        "24-25 LogLoss": log_losses[2],
        "25-26 LogLoss": log_losses[3]
    })


results_df = pd.DataFrame(results)

In [35]:
best = results_df.sort_values(
    ["MeanAccuracy", "NumFeatures"],
    ascending=[False, True]
)

print(
    best[
        [
            "NumFeatures",
            "Features",
            "MeanAccuracy",
            "AccuracyStd",
            "MeanLogLoss"
        ]
    ].head(20)
)

     NumFeatures                                           Features  \
231            4  EloDiff + ShotOTDiffLast5 + GoalAgainstDiffLas...   
229            3   EloDiff + ShotOTDiffLast5 + GoalAgainstDiffLast5   
217            3  EloDiff + ShotOTDiffLast5 + GoalsAgainstPerGam...   
225            4  EloDiff + ShotOTDiffLast5 + GoalsPerGameDiff +...   
197            3     EloDiff + ShotDiffLast5 + GoalAgainstDiffLast5   
158            4  EloDiff + GoalDiffLast5 + GoalsPerGameDiff + H...   
205            4  EloDiff + ShotDiffLast5 + GoalAgainstDiffLast5...   
230            4  EloDiff + ShotOTDiffLast5 + GoalAgainstDiffLas...   
311            3  PPGDiff + ShotDiffLast5 + HomeAwayGoalsAgainst...   
126            3       EloDiff + GoalsPerGameDiff + HomeAwayPPGDiff   
239            5  EloDiff + ShotOTDiffLast5 + GoalAgainstDiffLas...   
141            3  EloDiff + GoalAgainstDiffLast5 + GoalsPerGameDiff   
190            4  EloDiff + ShotDiffLast5 + GoalsPerGameDiff + H...   
241   

In [ ]:
features = [
    "EloDiff",
    "ShotOTDiffLast5",
    "GoalAgainstDiffLast5"
]